In [ ]:
import os
print(" Current working directory:", os.getcwd())

In [ ]:
import pandas as pd

# Load your input file
df = pd.read_csv('../../data/processed/Final_Model_Ready_Clean_Reshaped.csv')

# Convert 'Date' to datetime
df['Date'] = pd.to_datetime(df['Date'], format="%d-%m-%Y")

# Sort by Date
df = df.sort_values("Date").reset_index(drop=True)

# Add lag and rolling features
df['Wind_GWh_Lag1'] = df['Wind_GWh'].shift(1)
df['Wind_GWh_Lag2'] = df['Wind_GWh'].shift(2)
df['Wind_GWh_Lag12'] = df['Wind_GWh'].shift(12)
df['Solar_GWh_Lag1'] = df['Solar_GWh'].shift(1)
df['Solar_GWh_Lag2'] = df['Solar_GWh'].shift(2)
df['Solar_GWh_Lag12'] = df['Solar_GWh'].shift(12)

df['Wind_GWh_RollingMean3'] = df['Wind_GWh'].rolling(window=3).mean()
df['Solar_GWh_RollingMean3'] = df['Solar_GWh'].rolling(window=3).mean()

#  Merge TSO + DSO regional capacity (safely)
def safe_merge(df, col1, col2, new_col):
    if col1 in df.columns and col2 in df.columns:
        df[new_col] = df[col1] + df[col2]
    elif col1 in df.columns:
        df[new_col] = df[col1]
    elif col2 in df.columns:
        df[new_col] = df[col2]
    else:
        print(f" Skipped: {new_col} — neither {col1} nor {col2} found.")

# Applying safe_merge for each region/type
safe_merge(df, 'East_Wind_TSO_MW', 'East_Wind_DSO_MW', 'East_Wind_Total_MW')
safe_merge(df, 'West_Wind_TSO_MW', 'West_Wind_DSO_MW', 'West_Wind_Total_MW')
safe_merge(df, 'North_Wind_TSO_MW', 'North_Wind_DSO_MW', 'North_Wind_Total_MW')
safe_merge(df, 'South_Wind_TSO_MW', 'South_Wind_DSO_MW', 'South_Wind_Total_MW')
safe_merge(df, 'East_Solar_TSO_MW', 'East_Solar_DSO_MW', 'East_Solar_Total_MW')
safe_merge(df, 'West_Solar_TSO_MW', 'West_Solar_DSO_MW', 'West_Solar_Total_MW')
safe_merge(df, 'South_Solar_TSO_MW', 'South_Solar_DSO_MW', 'South_Solar_Total_MW')


tso_dso_cols = [col for col in df.columns if "_TSO_MW" in col or "_DSO_MW" in col]
df.drop(columns=tso_dso_cols, inplace=True)

# Dropping rows where lag or rolling features created NaNs
df = df.dropna().reset_index(drop=True)

# Save the final file
df.to_csv('../../data/processed/Final_Model_Ready_With_Lags_Rolling.csv', index=False)
print(" File saved as Final_Model_Ready_With_Lags_Rolling.csv")


<!-- Final Merge Strategy -->

Seprating the main final dataset to solar and wind for model training.

In [ ]:
import pandas as pd
import os

file_path = "../../data/processed/Final_Model_Ready_With_Lags_Rolling.csv"
df = pd.read_csv(file_path)

# Dropping rows with missing target values
df = df.dropna(subset=['Wind_GWh', 'Solar_GWh'])

# features for Wind model
wind_features = [
    'Date', 'Wind_GWh', 'Wind_GWh_Lag1', 'Wind_GWh_Lag2', 'Wind_GWh_Lag12', 'Wind_GWh_RollingMean3',
    'Wind_Speed_10m', 'Temperature_Celsius', 'Cloud_Cover',
    'Wind_Capacity_MW', 'North_Wind_Total_MW', 'South_Wind_Total_MW',
    'East_Wind_Total_MW', 'West_Wind_Total_MW'
]

# features for Solar model
solar_features = [
    'Date', 'Solar_GWh', 'Solar_GWh_Lag1', 'Solar_GWh_Lag2', 'Solar_GWh_Lag12', 'Solar_GWh_RollingMean3',
    'Solar_Radiation_MJ_per_m2', 'Temperature_Celsius', 'Cloud_Cover',
    'Solar_Capacity_MW','South_Solar_Total_MW', 'East_Solar_Total_MW', 'West_Solar_Total_MW'
]

# Subset the dataframes
df_wind = df[wind_features].copy()
df_solar = df[solar_features].copy()

# Save outputs
wind_path = "../../data/processed/Final_Model_Wind.csv"
solar_path = "../../data/processed/Final_Model_Solar.csv"

df_wind.to_csv(wind_path, index=False)
df_solar.to_csv(solar_path, index=False)


In [ ]:
#extended lag feature only for wind, used to run for ngboost variant model
import pandas as pd
import os

file_path = r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Wind_Data_Model.csv"
df = pd.read_csv(file_path, parse_dates=["Date"])
df = df.sort_values("Date").reset_index(drop=True)

#  Add New Lag Features 
df["Wind_GWh_Lag3"] = df["Wind_GWh"].shift(3)
df["Wind_GWh_Lag6"] = df["Wind_GWh"].shift(6)
df["Wind_GWh_Lag24"] = df["Wind_GWh"].shift(24)

# Add New Rolling Means
df["RollingMean_6"] = df["Wind_GWh"].rolling(window=6).mean().shift(1)
df["RollingMean_12"] = df["Wind_GWh"].rolling(window=12).mean().shift(1)

#  Drop NA Rows caused by shifting
df_cleaned = df.dropna().reset_index(drop=True)

output_path = r"C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Wind_Data_Model_With_Expanded_Lags.csv"
df_cleaned.to_csv(output_path, index=False)

print("Lag and rolling features added successfully!")
print("Cleaned file saved to:", output_path)
print("Final shape:", df_cleaned.shape)


Lag and rolling features added successfully!
Cleaned file saved to: C:\Projects\GitHub\Ireland-energy-forecast\data\processed\Final_Wind_Data_Model_With_Expanded_Lags.csv
Final shape: (144, 15)
